# Tự động hóa dự án đến kiểm tra chuyến đo

Phân vùng theo LEZ trước: A=thí điểm 2027; B=LEZ 2030 trừ A; C=ngoài LEZ 2030. Bộ 60 A20/B20/C20; bộ 30 A10/B8/C12. Bộ điều phối chạy cả kiểm tra độ nhạy mở rộng và dùng cache hợp lệ. Hướng dẫn: `work/docs/lez/PROJECT_AUTOMATION.md`.


In [1]:
from pathlib import Path
import sys, json, importlib
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates if (p / "scripts/lez/project_automation.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Mở notebook từ folder work có scripts/lez/project_automation.py.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.common import input_path

import lez.project_automation as automation
importlib.reload(automation)
preflight, run, status = automation.preflight, automation.run, automation.status

## 1. Kiểm tra cấu hình và dữ liệu

Đọc snapshot, hash nguồn, quota, phiên bản chính sách, runtime PyQGIS và cấu hình độ nhạy. Cell này không dựng hình học hay gửi API; baseline độ nhạy được kiểm tra sau khi bước chọn 30 hoàn tất.


In [2]:
CHECK = preflight(WORK)
print(json.dumps(CHECK, ensure_ascii=False, indent=2))

{
  "status": "ready",
  "candidate_quota": {
    "A": 20,
    "B": 20,
    "C": 20
  },
  "selection_quota": {
    "A_inside": 10,
    "B_inside": 8,
    "C_outside": 12
  },
  "matrix_cells": 12,
  "snapshot_utc": "2026-10-01T00:00:00Z",
  "network_calls": 0,
  "normalization_directory": "work/data/processed/normalization_2030/f42f05664165446b93ad",
  "qgis_runtime": "/Users/quanh.hg/python/.pyqgis-runtime/bin/python3.12",
  "zone_basis": "LEZ2027_pilot_and_LEZ2030_policy_first",
  "legacy_normalization_role": "Hanoi_source_and_supplemental_audit_only",
  "target_reallocation": {
    "basis": "2027 pilot eligible routing availability, preserve 20A and original scoring/RC criteria",
    "A": {
      "RC2": 6,
      "RC3": 7,
      "RC4": 7
    },
    "B": {
      "RC1": 5,
      "RC2": 5,
      "RC3": 5,
      "RC4": 5
    },
    "C": {
      "RC1": 5,
      "RC2": 5,
      "RC3": 5,
      "RC4": 5
    }
  },
  "routing_unavailable_cells": [
    {
      "sampling_zone": "A",
      "rc

## 2. Chạy toàn bộ

60 ứng viên → 30 đề xuất → độ nhạy mở rộng → bản đồ tên đường → biểu mẫu → bằng chứng → thuộc tính/chấm lại/thay tuyến → hành lang có hướng → GPS/tốc độ/chỉ tiêu vận hành.

`force=False` dùng cache có hash hợp lệ. Độ nhạy giữ quota A10/B8/C12, không thay bộ 30 công bố. `OBSERVATIONS_CSV=None` và `MEASUREMENTS_DIR=None` dùng dữ liệu đã có và folder mặc định; thiếu dữ liệu thực địa sẽ báo chờ. Chưa xây dựng chu trình lái.


In [ ]:
OBSERVATIONS_CSV = None
MEASUREMENTS_DIR = None
RESULT = run(WORK, force=False, progress=print,
             observations=OBSERVATIONS_CSV, input_dir=MEASUREMENTS_DIR)
print("Trạng thái:", RESULT["status"])
print("60/30:", RESULT["candidate_routes"], RESULT["selected_routes"])
print("Chuyến thực:", RESULT["measured_trips"])
print("Còn chờ:", RESULT["pending"])
print("Độ nhạy:", RESULT["sensitivity_completed_scenarios"], "/", RESULT["sensitivity_scenarios"],
      "; ca không đủ quota có chủ đích:", RESULT["sensitivity_expected_infeasible_scenarios"])


Đang xử lý: sampling_60
Đang xử lý: selection_30
Đang xử lý: selection_sensitivity
Đọc dữ liệu đã kiểm tra; tái lập baseline trước khi thử độ nhạy.
[1/114] baseline: completed; giữ 30/30; D=0.365187
[2/114] X_zscore: completed; giữ 30/30; D=0.365187
[3/114] weight_C1_minus20: completed; giữ 30/30; D=0.365187
[4/114] weight_C1_plus20: completed; giữ 30/30; D=0.365187
[5/114] weight_C2_minus20: completed; giữ 30/30; D=0.365187
[6/114] weight_C2_plus20: completed; giữ 30/30; D=0.365187
[7/114] weight_C3_minus20: completed; giữ 30/30; D=0.365187
[8/114] weight_C3_plus20: completed; giữ 30/30; D=0.365187
[9/114] weight_C4_minus20: completed; giữ 30/30; D=0.365187
[10/114] weight_C4_plus20: completed; giữ 30/30; D=0.365187
[11/114] weight_C5_minus20: completed; giữ 30/30; D=0.365187
[12/114] weight_C5_plus20: completed; giữ 30/30; D=0.365187
[13/114] weight_C6_minus20: completed; giữ 30/30; D=0.365187
[14/114] weight_C6_plus20: completed; giữ 30/30; D=0.365187
[15/114] weight_C7_minus20: com

## 3. Xem tiến độ và lỗi

Báo cáo lưu sau từng bước; `selection_sensitivity` là bước riêng. Thời gian trong bảng là thời gian thực của bước ở lượt điều phối này, gồm cả lượt dùng cache. Kịch bản chưa giải quyết chặn trạng thái hoàn tất; một ca kiểm tra cố ý thiếu quota được báo riêng.


In [ ]:
REPORT = status(WORK)
display(pd.DataFrame([
    {"step": key, "status": value.get("status"), "cache_hit": value.get("cache_hit"),
     "seconds_this_run": REPORT.get("step_seconds", {}).get(key),
     "completed_scenarios": value.get("completed_scenarios"),
     "expected_infeasible": value.get("expected_infeasible_scenarios"),
     "total_scenarios": value.get("total_scenarios")}
    for key, value in REPORT.get("steps", {}).items()
]))
print(json.dumps(REPORT.get("steps", {}).get("field_status", {}), ensure_ascii=False, indent=2))
print(json.dumps(REPORT.get("steps", {}).get("real_trip_qc", {}), ensure_ascii=False, indent=2))
print("Trạng thái:", REPORT.get("status"))
print("Bước đang chạy hoặc bị ngắt:", REPORT.get("active_step"))
print("Lỗi lần chạy:", REPORT.get("error"))


## 4. Kết quả độ nhạy mở rộng

Đọc kết quả của bước độ nhạy trong lượt điều phối vừa chạy. So sánh min–max/z-score, trọng số, ngưỡng ô, làn/tốc độ thiếu, C6 khác nhau và giả lập mất tuyến. Xem riêng trước/sau hoán đổi; tần suất kịch bản không phải xác suất chọn. Ca C6=0 là giả lập, không phải quan sát thực địa.

Chi tiết: `work/docs/lez/SENSITIVITY.md`.


In [ ]:
SENSITIVITY = REPORT.get("steps", {}).get("selection_sensitivity", {})
if not SENSITIVITY:
    print("Lượt điều phối chưa có bước độ nhạy mở rộng. Chạy lại mục 2.")
else:
    print(json.dumps({key: SENSITIVITY.get(key) for key in [
        "status", "cache_hit", "total_scenarios", "completed_scenarios",
        "expected_infeasible_scenarios", "unexpected_infeasible_scenarios",
        "baseline_reproduced_exactly", "scoring_stress_scenarios"
    ]}, ensure_ascii=False, indent=2))
    SENS_DIRECTORY = input_path(WORK, SENSITIVITY["paths"]["directory"])
    display(pd.read_csv(SENS_DIRECTORY / "families.csv", keep_default_na=False))
    display(pd.read_csv(SENS_DIRECTORY / "scenario_summary.csv", keep_default_na=False))
    display(pd.read_csv(SENS_DIRECTORY / "simulated_replacements.csv", keep_default_na=False))
    print("Kịch bản chưa giải quyết:", SENSITIVITY.get("unresolved", []))
    print("Báo cáo:", SENSITIVITY["paths"]["report"])


## 5. Tiếp tục sau khi dừng hoặc sau khi có dữ liệu

Chạy lại cell ở mục 2 với cùng cấu hình. Sau Interrupt/Ctrl+C tại độ nhạy, module giữ checkpoint từng kịch bản; chạy lại tiếp tục bằng cache đã kiểm hash. Báo cáo ghi bước bị ngắt hoặc lỗi, giữ các bước đã xong. Không cần `force=True` để tiếp tục.


In [ ]:
# Bỏ comment khi muốn tiếp tục:
# RESULT = run(WORK, force=False, progress=print,
#              observations=OBSERVATIONS_CSV, input_dir=MEASUREMENTS_DIR)
print("Báo cáo:", WORK / "reports/automation/latest_project_automation.json")

## 6. Các bản đồ và biểu mẫu

Bản đồ 30/60 và danh sách tên đường nằm trong `work/maps/routes`. Biểu mẫu/checklist và ưu tiên xác minh nằm trong bundle field_workflow. Ranh giới là proxy có nguồn và dải bất định; xác minh GIS chính thức và bằng chứng thực địa vẫn cần dữ liệu thực tế.


In [ ]:
print("Bản đồ:", WORK / "maps/routes")
FIELD = REPORT.get("steps", {}).get("field_templates", {})
print("Biểu mẫu:", FIELD.get("directory"))
print("Notebook nhập quan sát: work/notebooks/07_field_workflow.ipynb")
print("Notebook dữ liệu đo: work/notebooks/08_process_measurements.ipynb")